# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Look before deciding. Web traffic metrics are usually heavy-tailed: a few giant pages, a long tail of tiny ones. That is why the signal tests below use grouped bucket tables rather than raw correlations.

The key fields are:

- `recent30_impressions` — total March impressions
- `recent30_ctr_pct` — March clicks / impressions, already heavily skewed toward zero
- `recent30_avg_position` — March average position; zero or null means no position data
- `content_age_days` — days since content creation at the March 31 cutoff

In [12]:
%pip -q install duckdb huggingface_hub

import os
import getpass
from datetime import timedelta
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_ALL = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {FACT_DAILY}").fetchone()[0]
recent_start = cutoff_date - timedelta(days=29)
label_start = cutoff_date + timedelta(days=1)
label_end = cutoff_date + timedelta(days=30)

feature_frame = con.sql(f"""
    WITH recent AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS recent30_impressions,
            SUM(gsc_clicks) AS recent30_clicks,
            AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
            COUNT(DISTINCT report_date) AS recent30_days
        FROM {FACT_DAILY}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    future AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future30_impressions,
            COUNT(DISTINCT report_date) AS future30_days
        FROM {FACT_ALL}
        WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        r.client_hash_id,
        r.content_hash_id,
        r.recent30_impressions,
        LN(1 + r.recent30_impressions) AS log_recent30_impressions,
        100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
        r.recent30_avg_position,
        r.recent30_active_days,
        DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
        f.future30_impressions,
        CASE
            WHEN r.recent30_impressions >= 100
                 AND f.future30_impressions < 0.80 * r.recent30_impressions
            THEN 1 ELSE 0
        END AS is_declining_next30
    FROM recent r
    INNER JOIN future f USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} c USING (client_hash_id, content_hash_id)
    WHERE r.recent30_days >= 14
      AND f.future30_days >= 14
      AND r.recent30_impressions >= 100
""").df()

df = feature_frame.copy()

print(f"Feature frame rows: {len(df):,}")
print(f"Decline rate: {df['is_declining_next30'].mean():.3f}")
print(f"Rows with position data: {df['recent30_avg_position'].notna().sum():,}")

# Distributions -- heavy tails first
print("\nImpressions quantiles:")
print(df["recent30_impressions"].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(1).to_string())

print("\nCTR quantiles:")
print(df["recent30_ctr_pct"].quantile([0.05, 0.25, 0.5, 0.75, 0.9, 0.95]).round(2).to_string())

print("\nPosition quantiles (non-zero):")
print(df[df["recent30_avg_position"] > 0]["recent30_avg_position"].quantile([0.05, 0.25, 0.5, 0.75, 0.9, 0.95]).round(1).to_string())

print("\nDecline rate by content age bucket:")
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[0, 30, 90, 180, 365, np.inf],
    labels=["0-30d", "31-90d", "91-180d", "181-365d", "365+d"],
    right=False
)
print(df.groupby("age_bucket")["is_declining_next30"].agg(size="size", mean="mean").round(3).to_string())

print("\nDecline rate by position tier:")
df["position_tier"] = pd.cut(
    df["recent30_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["top_3", "top_10", "top_20", "deep"],
    right=False
)
print(df.groupby("position_tier")["is_declining_next30"].agg(size="size", mean="mean").round(3).to_string())

print("\nDecline rate by volume tier:")
df["volume_tier"] = np.where(df["recent30_impressions"] >= 500, "high_volume", "low_volume")
print(df.groupby("volume_tier")["is_declining_next30"].agg(size="size", mean="mean").round(3).to_string())


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Feature frame rows: 96,268
Decline rate: 0.511
Rows with position data: 96,268

Impressions quantiles:
0.25      311.0
0.50      858.0
0.75     2689.0
0.90     6721.0
0.95    11560.3
0.99    29978.0

CTR quantiles:
0.05    0.00
0.25    0.00
0.50    0.13
0.75    0.36
0.90    0.68
0.95    0.95

Position quantiles (non-zero):
0.05     2.5
0.25     5.1
0.50     9.0
0.75    19.7
0.90    33.3
0.95    44.1

Decline rate by content age bucket:
             size   mean
age_bucket              
0-30d        6720  0.277
31-90d      23725  0.467
91-180d     15772  0.615
181-365d    36374  0.549
365+d       13677  0.478

Decline rate by position tier:
                size   mean
position_tier              
top_3           7910  0.539
top_10         43882  0.499
top_20         20921  0.527
deep           23555  0.508

Decline rate by volume tier:
              size   mean
volume_tier              
high_volume  61029  0.501
low_volume 

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Three safe signals, each with one mini-test and a verdict: **CONFIRMED / OPPOSITE / MIXED / FALSE**.

- **Staleness**: pages 91+ days old should decline more than newer pages.
- **CTR vs. position**: among visible pages, top-10 pages with low CTR should decline more than top-10 pages with healthy CTR.
- **Volume**: high-traffic pages should show a different decline rate from low-traffic pages.

In [13]:
# Signal 1: Staleness
df["is_stale_91"] = df["content_age_days"] >= 91
s1 = df.groupby("is_stale_91")["is_declining_next30"].agg(n="size", decline_rate="mean").reset_index()
print("Signal 1: Staleness (>=91d)")
print(s1.to_string(index=False))
print("Verdict: CONFIRMED\n")

# Signal 2: CTR vs. position
visible = df[df["recent30_impressions"] >= 500].copy()
visible["top10_position"] = visible["recent30_avg_position"].between(1, 10)
visible["low_ctr"] = visible["recent30_ctr_pct"] < 1.0
s2 = (
    visible.groupby(["top10_position", "low_ctr"])["is_declining_next30"]
    .agg(n="size", decline_rate="mean")
    .reset_index()
    .sort_values(["top10_position", "low_ctr"])
)
print("Signal 2: CTR vs. position (within visible pages, impressions>=500)")
print(s2.to_string(index=False))
print("Verdict: CONFIRMED\n")

# Signal 3: Volume
df["high_volume"] = df["recent30_impressions"] >= 500
s3 = df.groupby("high_volume")["is_declining_next30"].agg(n="size", decline_rate="mean").reset_index()
print("Signal 3: Volume (high >= 500 impressions)")
print(s3.to_string(index=False))
print("Verdict: MIXED\n")

print("=" * 60)
print("Audit summary:")
print("  1. Staleness: CONFIRMED")
print("  2. CTR-vs-position: CONFIRMED")
print("  3. Volume: MIXED")
print("=" * 60)

Signal 1: Staleness (>=91d)
 is_stale_91     n  decline_rate
       False 30445      0.425390
        True 65823      0.550112
Verdict: CONFIRMED

Signal 2: CTR vs. position (within visible pages, impressions>=500)
 top10_position  low_ctr     n  decline_rate
          False    False   502      0.270916
          False     True 22461      0.530920
           True    False  1942      0.186406
           True     True 36124      0.503045
Verdict: CONFIRMED

Signal 3: Volume (high >= 500 impressions)
 high_volume     n  decline_rate
       False 35239      0.526859
        True 61029      0.501319
Verdict: MIXED

Audit summary:
  1. Staleness: CONFIRMED
  2. CTR-vs-position: CONFIRMED
  3. Volume: MIXED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's refresh flags lean on staleness: old pages are treated as more likely to need refresh. This reruns the staleness test on the flag-relevant slice — visible pages only, using the session's 180-day threshold.

In [14]:
# Flag-linked test: staleness behind refresh flags, rerun on visible slice
visible = df[df["recent30_impressions"] >= 500].copy()
visible["is_stale_180"] = visible["content_age_days"] >= 180

flag_bucket = (
    visible.groupby("is_stale_180")["is_declining_next30"]
    .agg(n="size", decline_rate="mean")
    .reset_index()
)

print("Flag-linked test: staleness (>=180d) within visible pages")
print(flag_bucket.to_string(index=False))
print("Verdict: CONFIRMED")

Flag-linked test: staleness (>=180d) within visible pages
 is_stale_180     n  decline_rate
        False 29774      0.478975
         True 31255      0.522604
Verdict: CONFIRMED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Staleness and CTR-vs-position are real, directional signals in this slice. Volume alone is not a decline trigger; it should be used as an editorial filter, not as a scoring input. The baseline should therefore score stale pages and low-CTR top-10 pages, while using visibility only to keep the queue editorially meaningful. These are observed correlations, not proof that refreshing fixes a page.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.